### 加载数据

In [14]:
import numpy as np
import polars as pl
import pandas as pd
from skfolio import Population,MultiPeriodPortfolio
from skfolio.datasets import load_sp500_dataset
from skfolio.preprocessing import prices_to_returns
from sklearn.model_selection import train_test_split
from log_result import init_logger, log_result, log_print, read_log
#from ml4t.data.providers.qmt_provider import QmtProvider
from sklearn import set_config
set_config(transform_output="pandas")
init_logger("WalkForward+ParameterSearch+Extremes")   

[log] 已绑定: D:\machine-learning-for-trading\case_studies\lazy_trading\logs\WalkForward+ParameterSearch+Extremes.log


WindowsPath('D:/machine-learning-for-trading/case_studies/lazy_trading/logs/WalkForward+ParameterSearch+Extremes.log')

In [2]:
# 加载数据
prices = pl.read_parquet("hs_funds_prices.parquet").to_pandas()
prices = prices.set_index('timestamp')
prices = prices.ffill()
prices = prices[prices.index.year > 2015]
all_symbols = prices.columns
# 市场基准
bench_symbol = "510300.SH"
bench = prices[bench_symbol]
prices = prices.drop(columns=[bench_symbol])  # 基准移出资产池
# 转换为线性收益率
X = prices_to_returns(prices, drop_inceptions_nan=False)
# 中性化
X_net = X.sub(prices_to_returns(bench.to_frame("bench"), drop_inceptions_nan=False)["bench"], axis=0)
# Inf值检查
inf_cols = X_net.columns[np.isinf(X_net).any(axis=0)]
print(inf_cols.tolist())
# 异常收益检查
X_net = X_net.drop(columns=(bad := X_net.columns[(X_net.abs() > 0.25).any()])); 
X = X.drop(columns=(bad := X.columns[(X.abs() > 0.25).any()])); 
print(f"数据异常：收益率超过±30%的列已剔除 {len(bad)} 列 -> {bad.tolist()}")

[]
数据异常：收益率超过±30%的列已剔除 3 列 -> ['161811.SZ', '510030.SH', '511580.SH']


### 预筛选
对初始资产宇宙进行预选择处理，其核心目的是在构建投资组合之前，通过特定的规则筛选出符合条件的资产，从而优化后续的计算效率和模型表现。

In [3]:
from wf_adaptive_search import load_adaptive_config
from wf_adaptive_search import adaptive_wf_search
from wf_cpcv_search import summarize_fold_params
from wf_cpcv_robustness import (perturbation_mc, sensitivity_curves, sensitivity_heatmap, topk_paths_eval,)
from wf_cpcv_ablation import ablate

In [5]:
cfg = load_adaptive_config("wf_adaptive_search_config.toml")

In [9]:
search_space = {'nondomin__min_n_assets': {'low': 5, 'high': 25, 'step': 5},
 'nondomin__threshold': {'low': -0.5, 'high': -0.3, 'step': 0.1},
 'correlate__threshold': {'low': 0.1, 'high': 0.5, 'step': 0.1},
 'extremes__k': {'low': 0.1, 'high': 0.5, 'step': 0.1},
 'nondomin__fitness_measures': ['mean-variance',
  'mean-variance-maxdd',
  'mean-variance-avgdd',
  'mean-semideviation-avgdd',
  'mean-mad-maxdd',
  'mean-mad-avgdd',
  'mean-mad-maxdd-cvar',
  'mean-mad-maxdd-cvar-sharpe',
  'mean-mad-avgdd-cvar',
  'mean-mad-avgdd-cvar-sharpe'],
 'train_size': {'low': 252, 'high': 504, 'step': 126}}

In [20]:
search_params = {'test_size': 252,
 'train_size': 756+252,
 'outer_purged_size': 1,
 'outer_reduce_test': True,
 'n_trials': 200,
 'sampler': 'random',
 'patience': 100,
 'min_delta': 0.0001,
 'n_jobs': 12,
 'seed': 42,
 'verbose': True}

In [21]:
folds = adaptive_wf_search(X, space=search_space, **search_params)

  0%|          | 0/200 [00:00<?, ?it/s]

Fold 0: IS score=4.7890 | test_ann=0.1023 | test_days=252


  0%|          | 0/200 [00:00<?, ?it/s]

Fold 1: IS score=9.3392 | test_ann=0.0169 | test_days=252


  0%|          | 0/200 [00:00<?, ?it/s]

...

Fold 4: IS score=3.2991 | test_ann=0.1313 | test_days=252


  0%|          | 0/200 [00:00<?, ?it/s]

Fold 5: IS score=4.5155 | test_ann=0.0312 | test_days=252


  0%|          | 0/200 [00:00<?, ?it/s]

Fold 6: IS score=9.6369 | test_ann=0.0117 | test_days=57


In [22]:
summarize_fold_params(folds)[['fold','score','dsr','sr_obs','margin','train ASR','test ASR']]                             # wf_cpcv_search

,fold,score,dsr,sr_obs,margin,train ASR,test ASR
0,0,4.7890,0.9999,4.7890,2.5116,2.4862,1.5845
1,1,9.3392,0.9999,9.3392,6.8625,8.9404,11.7086
2,2,2.4767,0.9999,2.4767,1.4693,1.9096,0.2632
3,3,2.2673,0.9999,2.2673,1.0339,1.1399,-0.6424
4,4,3.2991,0.9987,3.2991,1.2637,2.1674,1.4895
5,5,4.5155,0.9985,4.5155,1.6964,4.8133,0.6130
6,6,9.6369,0.9999,9.6369,7.0655,2.6873,10.5488


In [12]:
perturbation_mc(X, folds, space=cfg["space"])            # wf_cpcv_robustness

Fold 0: R=72 | fitness池=['mean-mad-maxdd-cvar', 'mean-mad-maxdd', 'mean-mad-avgdd-cvar'] | 扰动脉冲完成（72 组 × 普通OOS单测）
Fold 1: R=72 | fitness池=['mean-mad-avgdd-cvar', 'mean-mad-avgdd-cvar-sharpe', 'mean-variance-avgdd'] | 扰动脉冲完成（72 组 × 普通OOS单测）
Fold 2: R=72 | fitness池=['mean-semideviation-avgdd', 'mean-mad-avgdd', 'mean-mad-avgdd-cvar'] | 扰动脉冲完成（72 组 × 普通OOS单测）
Fold 3: R=72 | fitness池=['mean-semideviation-avgdd', 'mean-mad-avgdd-cvar', 'mean-mad-avgdd'] | 扰动脉冲完成（72 组 × 普通OOS单测）
Fold 4: R=72 | fitness池=['mean-mad-maxdd-cvar-sharpe', 'mean-mad-avgdd-cvar', 'mean-mad-avgdd'] | 扰动脉冲完成（72 组 × 普通OOS单测）
Fold 5: R=72 | fitness池=['mean-semideviation-avgdd', 'mean-mad-avgdd-cvar', 'mean-mad-avgdd-cvar-sharpe'] | 扰动脉冲完成（72 组 × 普通OOS单测）
Fold 6: R=72 | fitness池=['mean-mad-maxdd-cvar', 'mean-mad-maxdd', 'mean-variance'] | 扰动脉冲完成（72 组 × 普通OOS单测）


{'samples':              annualized_mean  annualized_sharpe_ratio  max_drawdown      skew  \
 fold sample                                                                     
 0    0              0.095732                 1.347410      0.055667 -0.184825   
      1              0.104320                 1.106674      0.065155 -0.272222   
      2              0.095732                 1.347410      0.055667 -0.184825   
      3              0.095732                 1.347410      0.055667 -0.184825   
      4              0.095732                 1.347410      0.055667 -0.184825   
 ...                      ...                      ...           ...       ...   
 6    67            -0.006315                -0.044587      0.098133  0.059615   
      68            -0.090086                -0.614311      0.099845  0.026885   
      69            -0.101635                -0.754866      0.078621 -0.039836   
      70            -0.006315                -0.044587      0.098133  0.059615   
     

In [ ]:
topk_paths_eval(X, folds, **cfg["paths_kwargs"])         # 压测段

In [24]:
frames, summary = ablate(X, folds)                # arms=None = 全部 5 臂
print(summary[["说明", "annualized_mean_mean", "annualized_sharpe_ratio_mean",
               "max_drawdown_mean", "样本数"]])            # wf_cpcv_ablation

Fold 0: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 1: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 2: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 3: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 4: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 5: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 0: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 1: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 2: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 3: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 4: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 5: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 0: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 1: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 2: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 3: paths=3 | window=504 | inner_folds=4 | test_days=252
Fold 4: paths=3 | window

In [ ]:
oos_pop = Population([fold['test'] for fold in folds])
oos_pop.plot_cumulative_returns()

In [ ]:
oos_mpt = MultiPeriodPortfolio([fold['test'] for fold in folds],compounded=True)
oos_mpt.plot_cumulative_returns()